In [1]:
from datasets import load_from_disk

ds = load_from_disk("../musiccaps/dataset_audio")
len(ds)

# preliminary results
# framework direkt metric output edicek
# machine translation metrics, BLEU, ROUGE, METEOR, CIDEr, bertscore

Loading dataset from disk:   0%|          | 0/19 [00:00<?, ?it/s]

5305

In [2]:
# First split: separate test set (held out for final evaluation only)
ds_split = ds.train_test_split(test_size=0.1, seed=42)
ds_train_val = ds_split["train"]
ds_test = ds_split["test"]

# Second split: split train+val into train and validation sets
ds_train_val_split = ds_train_val.train_test_split(test_size=0.1, seed=42)
ds_train = ds_train_val_split["train"]
ds_val = ds_train_val_split["test"]

print(f"Train: {len(ds_train)}, Val: {len(ds_val)}, Test: {len(ds_test)}")

Train: 4296, Val: 478, Test: 531


In [3]:
import torch# Cell 1: Imports
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ClapProcessor, ClapModel
import librosa
import torch
import torch.nn as nn
from pathlib import Path

In [4]:

# Cell 2: Device and Constants
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


Using device: cuda


In [5]:

# Constants (must match training)
D_AUDIO = 512  # CLAP projection dim
D_LM = 768     # GPT-2 embedding dim
PREFIX_LEN = 16  # Must match training


In [6]:

# Cell 3: Load Models
# CLAP (frozen)
clap_processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
clap = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device)
clap.eval()
for p in clap.parameters():
    p.requires_grad = False

# GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
gpt2.eval()
for p in gpt2.parameters():
    p.requires_grad = False


/home/aliozkaya/miniconda3/envs/musicgen/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:

# Cell 4: Define Projection Networks (must match training architecture)
projection = nn.Sequential(
    nn.Linear(D_AUDIO, D_LM * 2),      # 512 → 1536
    nn.LayerNorm(D_LM * 2),
    nn.GELU(),
    nn.Dropout(0.35),
    nn.Linear(D_LM * 2, D_LM * 4),     # 1536 → 3072
    nn.LayerNorm(D_LM * 4),
    nn.GELU(),
    nn.Dropout(0.35),
    nn.Linear(D_LM * 4, PREFIX_LEN * D_LM),  # 3072 → PREFIX_LEN * 768
).to(device)

text_projection = nn.Sequential(
    nn.Linear(D_LM, D_AUDIO),
    nn.LayerNorm(D_AUDIO),
    nn.Dropout(0.5),
    nn.GELU(),
).to(device)


In [8]:
# Cell 5: Load Checkpoint
SAVE_DIR = Path("../04_train/checkpoints")  # Adjust path as needed

checkpoint = torch.load(SAVE_DIR / "best_model_stage2.pt")

projection.load_state_dict(checkpoint["projection"])
text_projection.load_state_dict(checkpoint["text_projection"])
gpt2.load_state_dict(checkpoint["gpt2"])  # Load fine-tuned GPT-2

projection.eval()
text_projection.eval()
gpt2.eval()


print("Model loaded successfully!")

Model loaded successfully!


In [ ]:
# Cell 6: Helper Functions
def get_audio_embedding(sample):
    """Get audio embedding for a single sample"""
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    if sr != 48000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=48000)

    inputs = clap_processor(
        audios=audio,
        sampling_rate=48000,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        emb = clap.get_audio_features(**inputs)

    return emb  # (1, D_AUDIO)


In [ ]:
# Cell 7: Generation Function
def generate_caption(audio_sample, use_beam_search=True, max_length=100, min_length=20):
    """
    Generate caption from audio sample.
    
    Args:
        audio_sample: Dictionary with 'audio' key containing 'array' and 'sampling_rate'
        use_beam_search: If True, use beam search (better quality). If False, use sampling.
        max_length: Maximum length of generated caption (includes prefix)
        min_length: Minimum length of generated caption (includes prefix)
    
    Returns:
        Generated caption string
    """
    # Get audio embedding
    audio_emb = get_audio_embedding(audio_sample)
    
    # Project to prefix tokens
    prefix = projection(audio_emb)
    prefix = prefix.view(1, PREFIX_LEN, D_LM)
    
    # Get EOS token ID
    eos_token_id = tokenizer.eos_token_id
    pad_token_id = tokenizer.pad_token_id
    
    # Generate caption
    if use_beam_search:
        generated = gpt2.generate(
            inputs_embeds=prefix,
            max_length=max_length,
            min_length=min_length,
            num_beams=10,  # Balanced between quality and speed
            early_stopping=True,
            repetition_penalty=2.25,  # High penalty to prevent repetition
            no_repeat_ngram_size=2,  # Prevent 2-gram repetition (catches "passionate and passionate")
            length_penalty=0.8,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            do_sample=False,  # Deterministic with beam search
        )
    else:
        # Alternative: sampling with lower temperature
        generated = gpt2.generate(
            inputs_embeds=prefix,
            max_length=max_length,
            min_length=min_length,
            do_sample=True,
            temperature=0.3,  # Increased from 0.1 for more natural endings
            top_k=20,
            top_p=0.8,
            repetition_penalty=1.8,  # Increased to prevent repetition
            no_repeat_ngram_size=2,  # Prevent 2-gram repetition
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
        )
    
    # Decode to text - stop at EOS token
    caption = tokenizer.decode(generated[0], skip_special_tokens=True)
    
    # Additional cleanup: remove incomplete sentences at the end
    # Find last complete sentence (ends with . ! ?)
    last_period = max(caption.rfind('.'), caption.rfind('!'), caption.rfind('?'))
    if last_period > len(caption) * 0.5:  # Only if sentence is substantial
        caption = caption[:last_period + 1]
    
    return caption



In [52]:
sample = ds_test[20]
caption = generate_caption(sample, max_length=80, min_length=20)
print(f"Generated: {caption}")
print(f"Ground truth: {sample['caption']}")

Generated: The low quality recording features a pop song that consists of passionate male vocal singing over shimmering hi hats, groovy bass guitar melody and soft snare hits. The recording is noisy and it sounds emotional - like something you would hear in a romantic movie.
Ground truth: This song contains two acoustic guitars picking a melody. A digital drum is playing a simple groove when a male voice starts to sing in a higher register with backing vocals. Then the melody drops one octave. The guitars are panned to the left and right side of the speakers. This song may be playing on a road trip with friends.


In [53]:
from IPython.display import Audio
Audio(sample["audio"]["array"], rate=sample["audio"]["sampling_rate"])